<a href="https://colab.research.google.com/github/theeduazizjon-cell/computer_vision/blob/master/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Menu Detector Model!")

Menu Detector Model!


In [2]:
#----------------
#Import libraries
#----------------

from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#--------------------
# Define Dataset path
#--------------------

DATASET_PATH ='/content/drive/MyDrive/food101_dataset'

print("Dataset_path:", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_ckae':'dessert', #label group | class consolidation
    'cheesecake': 'dessert', #label group | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}

CLASSES =['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i , cls in enumerate(CLASSES)}
NUM_CLASS = len(CLASSES)

print("Num_class:", NUM_CLASS)
print("Class_to_idx:", CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
#pixel = (pixel-mean)/std




Dataset_path: /content/drive/MyDrive/food101_dataset
Num_class: 5
Class_to_idx: {'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [5]:
#--------------------
#Custom Dataset Class
#--------------------

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    label = self.labels[idx]
    try:
      image = Image.open(img_path).convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f'Skipping broken image: {img_path}')
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

In [6]:
#----------------------------
#Gathering and Splitting Data
#----------------------------

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print("class_path:", class_path)
  if not os.path.exists(class_path):
    print(f'Class path does not exist: {class_path}')
    continue
  for img in os.listdir(class_path):
    if img.endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

#print("all_images:", all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
imng, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_ckae
Class path does not exist: /content/drive/MyDrive/food101_dataset/chocolate_ckae
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
2476


In [7]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [9]:
#--------------------------------
#pretrained model --> MobileNetV2
#--------------------------------
model = mobilenet_v2(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASS) #fine-tunning in here

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [11]:
criterion = nn.CrossEntropyLoss() #loss function
optimizer = optim.Adam(model.parameters(), lr=0.001) #weight
torch.backends.cudnn.benchmark = True #benchmark setting

In [12]:
#-------------
#Training Loop
#-------------

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
  model.train() #train mode
  running_loss = 0.0# 70% | 30%Loss | 100%
  for images, labels in train_loader:# Forward and backward
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()#zero the gradient
    outputs = model(images)#Forward Pass
    loss = criterion(outputs, labels)#Calculate loss
    loss.backward()
    optimizer.step()#Adam optimizer
    running_loss += loss.item()#Track Loss

#Validation
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for images, labels in val_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total #Calculate Validation Accuracy
print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {running_loss/len(train_loader)}, Validation Accuracy: {val_acc}%')

if val_acc > best_accuracy:
  best_accuracy = val_acc
  torch.save(model.state_dict(), '/content/menu_detector.pth')
  print("Saved new best model!")

Epoch [10/10], Loss: 0.11363957899037558, Validation Accuracy: 86.45161290322581%
Saved new best model!
